# Se a Câmara realmente representasse o Brasil?
**Desvio Padrão (@desviopadrao.info)**

Compara o perfil dos 513 deputados federais eleitos com o da população brasileira em
gênero, raça, idade, escolaridade e ocupação. Renda e patrimônio ficam fora do escopo.

**Fontes**
- TSE — Repositório de Dados Eleitorais, `consulta_cand_{ano}.zip` (2018, 2022, 2026)
- IBGE — Censo Demográfico 2022, via SIDRA

**Notas metodológicas** (preenchidas ao longo do notebook)
- Gênero e cor/raça são **autodeclarados**, tanto no registro de candidatura (TSE) quanto no Censo (IBGE).


## 0. Configuração
Imports, caminhos e constantes. Os arquivos brutos ficam em `dados/` (cache local: só baixa se não existir)
e os JSONs finais em `output/`.

In [1]:
from pathlib import Path
import zipfile

import pandas as pd
import requests

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAIZ = Path.cwd()
DIR_TSE = RAIZ / "dados" / "tse"
DIR_IBGE = RAIZ / "dados" / "ibge"
DIR_OUT = RAIZ / "output"
for d in (DIR_TSE, DIR_IBGE, DIR_OUT):
    d.mkdir(parents=True, exist_ok=True)

ANOS = [2018, 2022, 2026]
ANO_PRINCIPAL = 2026  # base de 2026 já publicada com situação de eleito (verificado em 06/11/2026)
URL_TSE = "https://cdn.tse.jus.br/estatistica/sead/odsele/consulta_cand/consulta_cand_{ano}.zip"


## 1. Download e leitura das bases de candidatos do TSE
Para cada ano:
1. baixa `consulta_cand_{ano}.zip` (se ainda não estiver em `dados/tse/`);
2. lê o arquivo `consulta_cand_{ano}_BRASIL.csv` se existir; se não, concatena os CSVs das UFs
   (excluindo o `_BR`, que traz só candidaturas nacionais — presidente);
3. lê tudo como texto (`dtype=str`) para não perder zeros à esquerda em códigos.

Nesta etapa ainda **não** filtramos nada: só conferimos que as bases vieram inteiras.

In [2]:
def baixar_zip(ano: int) -> Path:
    """Baixa o zip do TSE para dados/tse/, se ainda não existir."""
    destino = DIR_TSE / f"consulta_cand_{ano}.zip"
    if not destino.exists():
        r = requests.get(URL_TSE.format(ano=ano), timeout=300)
        r.raise_for_status()
        destino.write_bytes(r.content)
    return destino


def ler_candidatos(ano: int) -> pd.DataFrame:
    """Lê o CSV _BRASIL do zip; se não houver, concatena os CSVs por UF."""
    caminho_zip = baixar_zip(ano)
    with zipfile.ZipFile(caminho_zip) as z:
        csvs = [n for n in z.namelist() if n.lower().endswith(".csv")]
        brasil = [n for n in csvs if n.upper().endswith("_BRASIL.CSV")]
        if brasil:
            arquivos = brasil
        else:
            # _BR = candidaturas nacionais (presidente); não interessa aqui
            arquivos = [n for n in csvs if not n.upper().endswith("_BR.CSV")]
        partes = []
        for nome in arquivos:
            with z.open(nome) as f:
                partes.append(pd.read_csv(f, sep=";", encoding="latin-1", dtype=str))
    df = pd.concat(partes, ignore_index=True)
    df.attrs["arquivos"] = arquivos
    return df


cand_raw = {ano: ler_candidatos(ano) for ano in ANOS}

resumo = pd.DataFrame({
    ano: {
        "arquivo(s) lido(s)": ", ".join(df.attrs["arquivos"]) if len(df.attrs["arquivos"]) == 1
                               else f"{len(df.attrs['arquivos'])} CSVs por UF",
        "linhas": len(df),
        "colunas": df.shape[1],
        "data de geração": df["DT_GERACAO"].iloc[0],
    }
    for ano, df in cand_raw.items()
}).T
resumo

,arquivo(s) lido(s),linhas,colunas,data de geração
2018,consulta_cand_2018_BRASIL.csv,29287,50,13/11/2024
2022,consulta_cand_2022_BRASIL.csv,29322,50,06/11/2026
2026,consulta_cand_2026_BRASIL.csv,21017,50,06/11/2026


In [3]:
# Cargos presentes e quantas linhas de Deputado Federal cada base tem, por situação de totalização
for ano, df in cand_raw.items():
    dep = df[df["DS_CARGO"].str.upper() == "DEPUTADO FEDERAL"]
    print(f"--- {ano}: {len(dep):,} linhas de Deputado Federal")
    print(dep["DS_SIT_TOT_TURNO"].value_counts().to_string(), "\n")

--- 2018: 8,607 linhas de Deputado Federal
DS_SIT_TOT_TURNO
SUPLENTE            4586
NÃO ELEITO          2984
#NULO#               524
ELEITO POR QP        387
ELEITO POR MÉDIA     126 

--- 2022: 10,630 linhas de Deputado Federal
DS_SIT_TOT_TURNO
NÃO ELEITO          5213
SUPLENTE            4244
#NULO                660
ELEITO POR QP        335
ELEITO POR MÉDIA     178 

--- 2026: 7,803 linhas de Deputado Federal
DS_SIT_TOT_TURNO
SUPLENTE            3712
NÃO ELEITO          3168
#NULO                410
ELEITO POR QP        385
ELEITO POR MÉDIA     128 



## 2. Deputados federais: candidatos e eleitos

**Recorte central:** a Câmara da legislatura **2027–2031** = os 513 eleitos em 2026.
2018 (legislatura 2019–2023) e 2022 (2023–2027) servem só para a evolução histórica.

O que esta etapa faz:
1. filtra apenas o cargo **Deputado Federal** e empilha os três anos numa tabela `dep`;
2. marca `ELEITO` (ELEITO POR QP ou ELEITO POR MÉDIA) e `APTO` (situação de candidatura = APTO) —
   a escolha do denominador do funil fica para a etapa do funil;
3. checa duplicatas de candidatura (`SQ_CANDIDATO`) dentro de cada ano;
4. calcula a **idade na posse**: 1º de fevereiro do ano seguinte à eleição (01/02/2027 para os eleitos de 2026);
5. valida 513 eleitos por ano e lista os valores distintos de cada variável.

> Nota: "eleitos" ≠ "quem efetivamente exerce o mandato". Licenças para cargos no Executivo, cassações e
> decisões judiciais posteriores mudam a composição. Usamos os eleitos segundo o TSE na data de geração da base.

In [4]:
COLUNAS = [
    "ANO_ELEICAO", "SG_UF", "SQ_CANDIDATO", "NM_URNA_CANDIDATO", "SG_PARTIDO",
    "DS_SITUACAO_CANDIDATURA", "DS_SIT_TOT_TURNO",
    "DS_GENERO", "DS_COR_RACA", "DT_NASCIMENTO", "DS_GRAU_INSTRUCAO", "DS_OCUPACAO",
]
SITUACOES_ELEITO = {"ELEITO POR QP", "ELEITO POR MÉDIA"}

dep = pd.concat(
    [df.loc[df["DS_CARGO"].str.upper() == "DEPUTADO FEDERAL", COLUNAS] for df in cand_raw.values()],
    ignore_index=True,
)
dep["ANO_ELEICAO"] = dep["ANO_ELEICAO"].astype(int)
dep["ELEITO"] = dep["DS_SIT_TOT_TURNO"].isin(SITUACOES_ELEITO)
dep["APTO"] = dep["DS_SITUACAO_CANDIDATURA"].str.upper().eq("APTO")

# Duplicatas: a mesma candidatura (SQ_CANDIDATO) aparecendo mais de uma vez no mesmo ano
dup = dep.duplicated(["ANO_ELEICAO", "SQ_CANDIDATO"], keep=False)
print("linhas duplicadas por (ano, SQ_CANDIDATO):", int(dup.sum()))

pd.crosstab(dep["ANO_ELEICAO"], dep["DS_SITUACAO_CANDIDATURA"], margins=True, margins_name="Total")

linhas duplicadas por (ano, SQ_CANDIDATO): 0


DS_SITUACAO_CANDIDATURA,#NE,APTO,INAPTO,Total
ANO_ELEICAO,,,,
2018,0,7641,966,8607
2022,0,9476,1154,10630
2026,7803,0,0,7803
Total,7803,17117,2120,27040


In [5]:
# Funil bruto por ano: registrados -> aptos -> eleitos
funil_bruto = dep.groupby("ANO_ELEICAO").agg(
    registrados=("SQ_CANDIDATO", "size"),
    aptos=("APTO", "sum"),
    eleitos=("ELEITO", "sum"),
)
# Algum eleito que não esteja como APTO? (ex.: sub judice)
funil_bruto["eleitos_nao_aptos"] = dep[dep["ELEITO"] & ~dep["APTO"]].groupby("ANO_ELEICAO").size()
funil_bruto = funil_bruto.fillna(0).astype(int)

assert (funil_bruto["eleitos"] == 513).all(), "Algum ano não tem 513 eleitos!"
funil_bruto

,registrados,aptos,eleitos,eleitos_nao_aptos
ANO_ELEICAO,,,,
2018,8607,7641,513,1
2022,10630,9476,513,1
2026,7803,0,513,513


In [6]:
# Idade na posse: 1º de fevereiro do ano seguinte à eleição
dep["DT_NASCIMENTO"] = pd.to_datetime(dep["DT_NASCIMENTO"], format="%d/%m/%Y", errors="coerce")
dep["DT_POSSE"] = pd.to_datetime((dep["ANO_ELEICAO"] + 1).astype(str) + "-02-01")

def idade_em(nasc: pd.Series, ref: pd.Series) -> pd.Series:
    """Idade em anos completos na data de referência."""
    fez_aniversario = (ref.dt.month > nasc.dt.month) | (
        (ref.dt.month == nasc.dt.month) & (ref.dt.day >= nasc.dt.day)
    )
    return ref.dt.year - nasc.dt.year - (~fez_aniversario).astype(int)

dep["IDADE_POSSE"] = idade_em(dep["DT_NASCIMENTO"], dep["DT_POSSE"])

eleitos = dep[dep["ELEITO"]].copy()
print("eleitos sem data de nascimento válida:", int(eleitos["DT_NASCIMENTO"].isna().sum()))
eleitos.groupby("ANO_ELEICAO")["IDADE_POSSE"].describe().round(1)

eleitos sem data de nascimento válida:

 0


,count,mean,std,min,25%,50%,75%,max
ANO_ELEICAO,,,,,,,,
2018,513.0,49.1,11.9,22.0,39.0,49.0,58.0,84.0
2022,513.0,49.8,12.3,21.0,40.0,49.0,59.0,88.0
2026,513.0,49.9,12.0,20.0,42.0,49.0,59.0,78.0


In [7]:
# Eleitos por UF (2026) — conferência da distribuição das bancadas
eleitos26 = eleitos[eleitos["ANO_ELEICAO"] == ANO_PRINCIPAL]
eleitos26["SG_UF"].value_counts().sort_index().to_frame("cadeiras").T

SG_UF,AC,AL,AM,AP,BA,CE,DF,ES,GO,MA,MG,MS,MT,PA,PB,PE,PI,PR,RJ,RN,RO,RR,RS,SC,SE,SP,TO
cadeiras,8,9,8,8,39,22,8,10,17,18,53,8,8,17,12,25,10,30,46,8,8,8,31,16,8,70,8


In [8]:
# Valores distintos das variáveis categóricas entre os ELEITOS, por ano
for var in ["DS_GENERO", "DS_COR_RACA", "DS_GRAU_INSTRUCAO"]:
    display(pd.crosstab(eleitos[var], eleitos["ANO_ELEICAO"], margins=True, margins_name="Total"))

ANO_ELEICAO,2018,2022,2026,Total
DS_GENERO,,,,
FEMININO,77,90,109,276
MASCULINO,436,423,404,1263
Total,513,513,513,1539


ANO_ELEICAO,2018,2022,2026,Total
DS_COR_RACA,,,,
AMARELA,2,3,1,6
BRANCA,387,366,374,1127
INDÍGENA,1,4,3,8
NÃO INFORMADO,0,1,0,1
PARDA,102,113,106,321
PRETA,21,26,29,76
Total,513,513,513,1539


ANO_ELEICAO,2018,2022,2026,Total
DS_GRAU_INSTRUCAO,,,,
ENSINO FUNDAMENTAL COMPLETO,10,10,8,28
ENSINO FUNDAMENTAL INCOMPLETO,5,4,5,14
ENSINO MÉDIO COMPLETO,41,27,36,104
ENSINO MÉDIO INCOMPLETO,2,2,3,7
LÊ E ESCREVE,1,0,0,1
SUPERIOR COMPLETO,417,426,421,1264
SUPERIOR INCOMPLETO,37,44,40,121
Total,513,513,513,1539


In [9]:
# Mesmas variáveis entre TODOS os candidatos registrados (base do funil), por ano
for var in ["DS_GENERO", "DS_COR_RACA"]:
    display(pd.crosstab(dep[var], dep["ANO_ELEICAO"], margins=True, margins_name="Total"))

ANO_ELEICAO,2018,2022,2026,Total
DS_GENERO,,,,
FEMININO,2767,3717,2871,9355
MASCULINO,5821,6905,4932,17658
NÃO DIVULGÁVEL,19,8,0,27
Total,8607,10630,7803,27040


ANO_ELEICAO,2018,2022,2026,Total
DS_COR_RACA,,,,
AMARELA,59,44,44,147
BRANCA,4904,5327,4034,14265
INDÍGENA,39,59,74,172
NÃO DIVULGÁVEL,19,8,0,27
NÃO INFORMADO,0,60,0,60
PARDA,2649,3646,2601,8896
PRETA,937,1486,1050,3473
Total,8607,10630,7803,27040


In [10]:
# Ocupação declarada: número de categorias distintas e top 15 entre os eleitos de 2026
print("ocupações distintas entre os eleitos de 2026:", eleitos26["DS_OCUPACAO"].nunique())
eleitos26["DS_OCUPACAO"].value_counts().head(15).to_frame("eleitos")

ocupações distintas entre os eleitos de 2026: 49


,eleitos
DS_OCUPACAO,
DEPUTADA/DEPUTADO,201
EMPRESÁRIA/EMPRESÁRIO,61
ADVOGADA/ADVOGADO,55
MÉDICA/MÉDICO,30
VEREADORA/VEREADOR,27
ADMINISTRADORA/ADMINISTRADOR,10
SERVIDOR PÚBLICO ESTADUAL,9
OUTROS,9
PROFESSORA/PROFESSOR DE ENSINO SUPERIOR,8


### 2.1 Decisão: universo de candidatos do funil
Em 2026 o TSE não preencheu `DS_SITUACAO_CANDIDATURA` (todas as linhas vêm como `#NE`), então não dá
para filtrar "aptos" de forma comparável entre os anos.

**Decisão:** candidato do funil = candidatura a deputado federal cujo `DS_SIT_TOT_TURNO` **não** é `#NULO`
(ou `#NULO#`, grafia de 2018), isto é, candidaturas cujos votos foram totalizados.

Limitações registradas:
- em 2018/2022, esse critério não equivale exatamente a "apto": candidatos INAPTOS que aparecem como
  "NÃO ELEITO" (fora do `#NULO`) ficam dentro do funil;
- o `#NULO` de 2026 ainda pode mudar com julgamentos pendentes.

In [11]:
dep["CANDIDATO_FUNIL"] = ~dep["DS_SIT_TOT_TURNO"].str.upper().str.startswith("#NULO")

dep.groupby("ANO_ELEICAO").agg(
    registrados=("SQ_CANDIDATO", "size"),
    candidatos_funil=("CANDIDATO_FUNIL", "sum"),
    eleitos=("ELEITO", "sum"),
)

,registrados,candidatos_funil,eleitos
ANO_ELEICAO,,,
2018,8607,8083,513
2022,10630,9970,513
2026,7803,7393,513


## 3. IBGE — Censo 2022: catálogo das tabelas do SIDRA
Antes de baixar qualquer dado, listamos as tabelas candidatas e seus metadados
(variáveis, classificações e categorias), consultando a API de agregados do IBGE.

| Tema | Tabela | Universo | Uso previsto |
|---|---|---|---|
| Sexo × cor/raça × idade | **9606** | População residente | % de mulheres, % por cor/raça e pirâmide etária 18+ (idade simples, até "100 anos ou mais") |
| Escolaridade | **10061** | Pessoas de 18 anos ou mais (com recorte "25 anos ou mais" na própria tabela) | Nível de instrução da população adulta |
| Ocupação | 10264 | Ocupados de 14 anos ou mais, por grandes grupos da COD | Só se houver comparação de ocupação (ver decisão pendente) |

Alternativas consideradas e descartadas: 9514 (sexo × idade, sem cor/raça — a 9606 já cobre),
10141/10142 (escolaridade 25+ por sexo ou por raça — a 10061 já traz o recorte 25+).

In [12]:
URL_META = "https://servicodados.ibge.gov.br/api/v3/agregados/{tabela}/metadados"
TABELAS_SIDRA = {9606: "sexo × cor/raça × idade", 10061: "escolaridade 18+", 10264: "ocupação (grandes grupos)"}

def resumo_metadados(tabela: int) -> None:
    m = requests.get(URL_META.format(tabela=tabela), timeout=60).json()
    print(f"### {tabela} — {m['nome']}")
    print("    pesquisa:", m["pesquisa"], "| níveis:", ", ".join(m["nivelTerritorial"]["Administrativo"]))
    for v in m["variaveis"]:
        print(f"    variável {v['id']}: {v['nome']} [{v['unidade']}]")
    for c in m["classificacoes"]:
        cats = [k["nome"] for k in c["categorias"]]
        amostra = "; ".join(cats if len(cats) <= 13 else cats[:6] + ["…"] + cats[-3:])
        print(f"    classificação {c['id']} — {c['nome']} ({len(cats)} categorias): {amostra}")
    print()

for t in TABELAS_SIDRA:
    resumo_metadados(t)

### 9606 — População residente, por cor ou raça, segundo o sexo e a idade
    pesquisa: Censo Demográfico | níveis: N1, N2, N6, N3
    variável 93: População residente [Pessoas]
    variável 1000093: População residente - percentual do total geral [%]
    classificação 86 — Cor ou raça (6 categorias): Total; Branca; Preta; Amarela; Parda; Indígena
    classificação 2 — Sexo (3 categorias): Total; Homens; Mulheres
    classificação 287 — Idade (134 categorias): Total; 0 a 4 anos; Menos de 1 ano; Menos de 1 mês; 1 mês; 2 meses; …; 98 anos; 99 anos; 100 anos ou mais



### 10061 — Pessoas de 18 anos ou mais de idade, por nível de instrução, segundo os grupos de idade, o sexo e a cor ou raça
    pesquisa: Censo Demográfico | níveis: N1, N2, N6, N3
    variável 2667: Pessoas de 18 anos ou mais de idade [Pessoas]
    variável 1002667: Pessoas de 18 anos ou mais de idade - percentual do total geral [%]
    classificação 1568 — Nível de instrução (5 categorias): Total; Sem instrução e fundamental incompleto; Fundamental completo e médio incompleto; Médio completo e superior incompleto; Superior completo
    classificação 58 — Grupo de idade (19 categorias): Total; 18 a 24 anos; 18 a 19 anos; 20 a 24 anos; 25 anos ou mais; 25 a 64 anos; …; 70 a 74 anos; 75 a 79 anos; 80 anos ou mais
    classificação 2 — Sexo (3 categorias): Total; Homens; Mulheres
    classificação 86 — Cor ou raça (6 categorias): Total; Branca; Preta; Amarela; Parda; Indígena



### 10264 — Pessoas de 14 anos ou mais de idade ocupadas na semana de referência, por sexo, cor ou raça, condição de contribuição para instituto de previdência oficial no trabalho principal e grandes grupos de ocupação no trabalho principal
    pesquisa: Censo Demográfico | níveis: N1, N2, N6, N3
    variável 4090: Pessoas de 14 anos ou mais de idade ocupadas na semana de referência [Pessoas]
    variável 1004090: Pessoas de 14 anos ou mais de idade ocupadas na semana de referência - percentual do total geral [%]
    classificação 2 — Sexo (3 categorias): Total; Homens; Mulheres
    classificação 86 — Cor ou raça (6 categorias): Total; Branca; Preta; Amarela; Parda; Indígena
    classificação 526 — Condição de contribuição para instituto de previdência oficial no trabalho principal (3 categorias): Total; Contribuintes; Não contribuintes
    classificação 12064 — Grandes grupos de ocupação no trabalho principal (12 categorias): Total; Diretores e gerentes; Profissionais das ciências e i

## 4. IBGE — download das tabelas do Censo 2022

**Decisões registradas**
- Escolaridade: população de **25 anos ou mais** (padrão do IBGE para escolaridade concluída; entre 18 e 24
  anos muita gente ainda está estudando). Vem da tabela 10061, categoria "25 anos ou mais".
- Gênero e cor/raça: população residente **total** (tabela 9606). Idade: população de **18 anos ou mais** (9606).
- Referência temporal: o Censo retrata a população em 2022; a Câmara toma posse em 2027.

O que esta etapa faz:
1. baixa as tabelas 9606 e 10061 inteiras (todas as categorias), nível Brasil, via `sidrapy`,
   com cache em `dados/ibge/`;
2. limpa o formato do SIDRA (1ª linha = cabeçalho) e converte `Valor` para número;
3. confere consistência interna: as partes somam o total? O total de 18+ bate entre as duas tabelas?

In [13]:
import sidrapy

def baixar_sidra(nome: str, **kwargs) -> pd.DataFrame:
    """Baixa uma tabela do SIDRA (nível Brasil, 2022), limpa e guarda em cache CSV."""
    cache = DIR_IBGE / f"{nome}.csv"
    if cache.exists():
        return pd.read_csv(cache, dtype={"Valor": "float64"})
    bruto = sidrapy.get_table(territorial_level="1", ibge_territorial_code="all", period="2022", **kwargs)
    df = bruto.iloc[1:].copy()
    df.columns = bruto.iloc[0]
    # SIDRA usa "-" para zero absoluto; "..." / "X" para não disponível ou sigiloso
    df["Valor"] = pd.to_numeric(df["Valor"].replace("-", "0"), errors="coerce")
    df = df.reset_index(drop=True)
    df.to_csv(cache, index=False)
    return df

# 9606: população residente por cor ou raça, sexo e idade
pop = baixar_sidra(
    "sidra_9606_sexo_raca_idade", table_code="9606", variable="93",
    classifications={"86": "all", "2": "all", "287": "all"},
)
# 10061: pessoas de 18+ por nível de instrução, grupo de idade, sexo e cor ou raça
esc = baixar_sidra(
    "sidra_10061_instrucao_18mais", table_code="10061", variable="2667",
    classifications={"1568": "all", "58": "all", "2": "all", "86": "all"},
)
print("9606:", pop.shape, "| valores nulos:", int(pop["Valor"].isna().sum()))
print("10061:", esc.shape, "| valores nulos:", int(esc["Valor"].isna().sum()))
pop[["Cor ou raça", "Sexo", "Idade", "Valor"]].head()

9606: (2412, 17) | valores nulos: 0
10061: (1710, 19) | valores nulos: 0


,Cor ou raça,Sexo,Idade,Valor
0,Total,Total,Total,203080756.0
1,Total,Total,0 a 4 anos,12704860.0
2,Total,Total,Menos de 1 ano,2358265.0
3,Total,Total,Menos de 1 mês,185145.0
4,Total,Total,1 mês,186440.0


In [14]:
# --- Checagens de consistência da 9606 ---
import re

def sel(df, **filtros):
    """Filtra o DataFrame do SIDRA por nome de categoria: sel(pop, Sexo='Total', ...)."""
    m = pd.Series(True, index=df.index)
    for col, val in filtros.items():
        col = col.replace("_", " ")
        m &= df[col].isin(val) if isinstance(val, (list, set)) else df[col].eq(val)
    return df[m]

total_br = sel(pop, **{"Cor_ou_raça": "Total", "Sexo": "Total", "Idade": "Total"})["Valor"].item()

# Idades simples (0 a 99 anos + "100 anos ou mais"), sem os grupos quinquenais e sem os meses
padrao_idade_simples = re.compile(r"^(Menos de 1 ano|\d+ anos?|100 anos ou mais)$")
pop["IDADE_SIMPLES"] = pop["Idade"].str.match(padrao_idade_simples)
pop["IDADE_NUM"] = pd.to_numeric(
    pop["Idade"].str.extract(r"^(\d+)")[0].where(pop["IDADE_SIMPLES"]), errors="coerce"
)
pop.loc[pop["Idade"].eq("Menos de 1 ano"), "IDADE_NUM"] = 0

base = sel(pop, **{"Cor_ou_raça": "Total", "Sexo": "Total"})
checagens = {
    "população total (Total/Total/Total)": total_br,
    "soma das cores/raças": sel(pop, Sexo="Total", Idade="Total").query("`Cor ou raça` != 'Total'")["Valor"].sum(),
    "soma dos sexos": sel(pop, **{"Cor_ou_raça": "Total", "Idade": "Total"}).query("Sexo != 'Total'")["Valor"].sum(),
    "soma das idades simples": base.loc[base["IDADE_SIMPLES"], "Valor"].sum(),
}
pd.Series(checagens, name="pessoas").map("{:,.0f}".format).to_frame()

,pessoas
população total (Total/Total/Total),"203,080,756"
soma das cores/raças,"203,069,637"
soma dos sexos,"203,080,756"
soma das idades simples,"203,080,756"


In [15]:
# --- Checagens de consistência da 10061 e cruzamento com a 9606 ---
pop_18mais_9606 = base.loc[base["IDADE_NUM"] >= 18, "Valor"].sum()
pop_25mais_9606 = base.loc[base["IDADE_NUM"] >= 25, "Valor"].sum()

esc_tot = sel(esc, Sexo="Total", **{"Cor_ou_raça": "Total"})
esc_18 = sel(esc_tot, **{"Grupo_de_idade": "Total", "Nível_de_instrução": "Total"})["Valor"].item()
esc_25 = sel(esc_tot, **{"Grupo_de_idade": "25 anos ou mais", "Nível_de_instrução": "Total"})["Valor"].item()
esc_25_partes = sel(esc_tot, **{"Grupo_de_idade": "25 anos ou mais"}).query("`Nível de instrução` != 'Total'")["Valor"].sum()

pd.DataFrame({
    "9606 (idades simples)": [pop_18mais_9606, pop_25mais_9606],
    "10061 (total)": [esc_18, esc_25],
    "10061 (soma dos níveis)": [None, esc_25_partes],
}, index=["18 anos ou mais", "25 anos ou mais"]).map(lambda v: "" if pd.isna(v) else f"{v:,.0f}")

,9606 (idades simples),10061 (total),10061 (soma dos níveis)
18 anos ou mais,"154,346,198","154,309,487",
25 anos ou mais,"133,109,090","133,108,020","133,108,021"


In [16]:
# Prévia (ainda sem harmonizar): distribuição da população por sexo e cor/raça (total) e escolaridade 25+
prev_sexo = sel(pop, **{"Cor_ou_raça": "Total", "Idade": "Total"}).set_index("Sexo")["Valor"]
prev_raca = sel(pop, Sexo="Total", Idade="Total").set_index("Cor ou raça")["Valor"]
prev_esc = sel(esc_tot, **{"Grupo_de_idade": "25 anos ou mais"}).set_index("Nível de instrução")["Valor"]

for titulo, s in [("Sexo — população total", prev_sexo), ("Cor ou raça — população total", prev_raca),
                  ("Nível de instrução — 25 anos ou mais", prev_esc)]:
    t = s.to_frame("pessoas")
    t["%"] = (100 * t["pessoas"] / s["Total"]).round(1)
    t["pessoas"] = t["pessoas"].map("{:,.0f}".format)
    print(titulo); display(t)

Sexo — população total


,pessoas,%
Sexo,,
Total,"203,080,756",100.0
Homens,"98,532,431",48.5
Mulheres,"104,548,325",51.5


Cor ou raça — população total


,pessoas,%
Cor ou raça,,
Total,"203,080,756",100.0
Branca,"88,252,121",43.5
Preta,"20,656,458",10.2
Amarela,"850,130",0.4
Parda,"92,083,286",45.3
Indígena,"1,227,642",0.6


Nível de instrução — 25 anos ou mais


,pessoas,%
Nível de instrução,,
Total,"133,108,020",100.0
Sem instrução e fundamental incompleto,"46,912,823",35.2
Fundamental completo e médio incompleto,"18,666,167",14.0
Médio completo e superior incompleto,"43,044,784",32.3
Superior completo,"24,484,247",18.4


## 5. Ocupação: mapeamento TSE → grandes grupos da COD (Censo 2022)

O Censo classifica a ocupação pela **COD** (Classificação de Ocupações para Pesquisas Domiciliares, derivada da
CIUO-08/ISCO-08), com 10 grandes grupos. O TSE usa lista própria, **autodeclarada** no registro de candidatura.
Não existe tabela oficial de correspondência: o mapeamento abaixo é **nosso**, ocupação por ocupação, seguindo a
lógica da COD. Cada linha traz uma nota quando a escolha é ambígua.

Regras gerais adotadas:
- **Mandatos políticos** (deputado, vereador, senador, ministro de Estado) → grupo 1 na COD (legisladores e
  dirigentes do poder público), mas marcados à parte (`POLITICO=True`) para a decisão de tratamento.
- **Profissões de nível superior** (advogado, médico, engenheiro, professor, jornalista…) → grupo 2.
- **Policial militar e Forças Armadas** → grupo 0; **policial civil** → grupo 5 (na COD, o grupo 0 é só militar).
- **Sem ocupação identificável** (servidor público sem cargo, "outros") e **fora da força de trabalho**
  (aposentado, estudante, dona de casa, militar reformado) → `NÃO CLASSIFICÁVEL`, fora do denominador.

População de comparação: **pessoas de 14 anos ou mais ocupadas** na semana de referência (tabela 10264).

In [17]:
# Grandes grupos da COD (nomes exatamente como no SIDRA, tabela 10264, classificação 12064)
G0 = "Membros das forças armadas, policiais, bombeiros militares"
G1 = "Diretores e gerentes"
G2 = "Profissionais das ciências e intelectuais"
G3 = "Técnicos e profissionais de nível médio"
G4 = "Trabalhadores de apoio administrativo"
G5 = "Trabalhadores dos serviços, vendedores dos comércios e mercados"
G6 = "Trabalhadores qualificados da agropecuária, florestais, da caça e da pesca"
G9 = "Ocupações elementares"
NC = "NÃO CLASSIFICÁVEL"

# ocupação TSE -> (grande grupo COD, nota sobre a escolha; "" quando não há ambiguidade)
MAPA_OCUPACAO_COD = {
    # --- mandatos políticos (grupo 1 na COD; tratados à parte) ---
    "DEPUTADA/DEPUTADO": (G1, "mandato político"),
    "VEREADORA/VEREADOR": (G1, "mandato político"),
    "SENADORA/SENADOR": (G1, "mandato político"),
    "MINISTRA/MINISTRO DE ESTADO": (G1, "mandato político (dirigente do poder público)"),
    # --- grupo 1 ---
    "EMPRESÁRIA/EMPRESÁRIO": (G1, "ambíguo: COD classifica pelo que a pessoa faz; assumimos dirigente da própria empresa"),
    # --- grupo 2: profissionais das ciências e intelectuais ---
    "ADVOGADA/ADVOGADO": (G2, ""),
    "MÉDICA/MÉDICO": (G2, ""),
    "ADMINISTRADORA/ADMINISTRADOR": (G2, "ambíguo com grupo 1; tratado como profissão de nível superior (analista de gestão)"),
    "ENGENHEIRA/ENGENHEIRO": (G2, ""),
    "PROFESSORA/PROFESSOR DE ENSINO SUPERIOR": (G2, ""),
    "PROFESSORA/PROFESSOR DE ENSINO MÉDIO": (G2, ""),
    "JORNALISTA E REDATORA/REDATOR": (G2, ""),
    "ECONOMISTA": (G2, ""),
    "ENFERMEIRA/ENFERMEIRO": (G2, ""),
    "ODONTÓLOGA/ODONTÓLOGO": (G2, ""),
    "HISTORIADORA/HISTORIADOR": (G2, ""),
    "AGRÔNOMA/AGRÔNOMO": (G2, ""),
    "CONTADORA/CONTADOR": (G2, ""),
    "VETERINÁRIA/VETERINÁRIO": (G2, ""),
    "RELAÇÕES-PÚBLICAS": (G2, ""),
    "FARMACÊUTICA/FARMACÊUTICO": (G2, ""),
    "FISIOTERAPEUTA E TERAPEUTA OCUPACIONAL": (G2, ""),
    "PSICÓLOGA/PSICÓLOGO": (G2, ""),
    "ARQUITETA/ARQUITETO": (G2, ""),
    "ATRIZ/ATOR E DIRETORA/DIRETOR DE ESPETÁCULOS PÚBLICOS": (G2, ""),
    "PUBLICITÁRIA/PUBLICITÁRIO": (G2, ""),
    "ESCRITORA/ESCRITOR E CRÍTICA/CRÍTICO": (G2, ""),
    "LOCUTORA/LOCUTOR E COMENTARISTA DE RÁDIO E TELEVISÃO E RADIALISTA": (G2, ""),
    "SACERDOTE/SACERDOTISA OU MEMBRA/MEMBRO DE ORDEM OU SEITA RELIGIOSA": (G2, "COD: profissionais religiosos"),
    # --- grupo 3: técnicos e profissionais de nível médio ---
    "AGENTE DE SAÚDE E SANITARISTA": (G3, "ambíguo: sanitarista de nível superior seria grupo 2"),
    "ATLETA PROFISSIONAL E TÉCNICA/TÉCNICO EM DESPORTOS": (G3, ""),
    "FOTÓGRAFA/FOTÓGRAFO E ASSEMELHADOS": (G3, ""),
    # --- grupo 4: apoio administrativo ---
    "BANCÁRIA/BANCÁRIO E ECONOMIÁRIA/ECONOMIÁRIO": (G4, "ambíguo: cargo bancário não especificado"),
    # --- grupo 5: serviços e comércio ---
    "COMERCIANTE": (G5, "ambíguo com grupo 1 (gerente de comércio); tratado como dono de pequeno comércio"),
    "VENDEDORA/VENDEDOR DE COMÉRCIO VAREJISTA E ATACADISTA": (G5, ""),
    "POLICIAL CIVIL": (G5, "na COD, policial civil fica no grupo 5 (serviços de proteção)"),
    # --- grupo 6: agropecuária ---
    "PRODUTORA/PRODUTOR AGROPECUÁRIO": (G6, "ambíguo: grande produtor poderia ser grupo 1 (gerente agropecuário)"),
    "AGRICULTORA/AGRICULTOR": (G6, ""),
    "PECUARISTA": (G6, "ambíguo: idem produtor agropecuário"),
    # --- grupo 9: ocupações elementares ---
    "TRABALHADORA/TRABALHADOR RURAL": (G9, "ambíguo com grupo 6; tratado como trabalhador agropecuário assalariado"),
    # --- grupo 0: militares ---
    "POLICIAL MILITAR": (G0, ""),
    "MEMBRA/MEMBRO DAS FORÇAS ARMADAS": (G0, ""),
    # --- sem correspondência ---
    "SERVIDOR PÚBLICO ESTADUAL": (NC, "cargo não informado"),
    "SERVIDOR PÚBLICO FEDERAL": (NC, "cargo não informado"),
    "OUTROS": (NC, "ocupação não informada"),
    "ESTUDANTE, BOLSISTA, ESTAGIÁRIA/ESTAGIÁRIO E ASSEMELHADOS": (NC, "fora da população ocupada"),
    "APOSENTADA/APOSENTADO (EXCETO SERVIDOR PÚBLICO/SERVIDORA PÚBLICA)": (NC, "fora da população ocupada"),
    "DONA/DONO DE CASA": (NC, "fora da população ocupada"),
    "MILITAR REFORMADA/REFORMADO": (NC, "fora da população ocupada"),
}
OCUPACOES_POLITICAS = {"DEPUTADA/DEPUTADO", "VEREADORA/VEREADOR", "SENADORA/SENADOR", "MINISTRA/MINISTRO DE ESTADO"}

eleitos26 = eleitos[eleitos["ANO_ELEICAO"] == ANO_PRINCIPAL].copy()
faltando = set(eleitos26["DS_OCUPACAO"]) - set(MAPA_OCUPACAO_COD)
assert not faltando, f"Ocupações sem mapeamento: {faltando}"

eleitos26["COD_GRUPO"] = eleitos26["DS_OCUPACAO"].map(lambda o: MAPA_OCUPACAO_COD[o][0])
eleitos26["POLITICO"] = eleitos26["DS_OCUPACAO"].isin(OCUPACOES_POLITICAS)

# Tabela de revisão do mapeamento: ocupação, nº de eleitos, grupo e nota
revisao = (eleitos26.groupby("DS_OCUPACAO").size().rename("eleitos").to_frame()
           .assign(grupo_COD=lambda t: [MAPA_OCUPACAO_COD[o][0] for o in t.index],
                   nota=lambda t: [MAPA_OCUPACAO_COD[o][1] for o in t.index])
           .sort_values(["grupo_COD", "eleitos"], ascending=[True, False]))
revisao

,eleitos,grupo_COD,nota
DS_OCUPACAO,,,
DEPUTADA/DEPUTADO,201,Diretores e gerentes,mandato político
EMPRESÁRIA/EMPRESÁRIO,61,Diretores e gerentes,ambíguo: COD classifica pelo que a pessoa faz;...
VEREADORA/VEREADOR,27,Diretores e gerentes,mandato político
MINISTRA/MINISTRO DE ESTADO,1,Diretores e gerentes,mandato político (dirigente do poder público)
SENADORA/SENADOR,1,Diretores e gerentes,mandato político
POLICIAL MILITAR,7,"Membros das forças armadas, policiais, bombeir...",
MEMBRA/MEMBRO DAS FORÇAS ARMADAS,2,"Membros das forças armadas, policiais, bombeir...",
OUTROS,9,NÃO CLASSIFICÁVEL,ocupação não informada
SERVIDOR PÚBLICO ESTADUAL,9,NÃO CLASSIFICÁVEL,cargo não informado


In [18]:
# 10264: pessoas de 14+ ocupadas por grande grupo de ocupação (Total de sexo, cor/raça e contribuição)
ocup = baixar_sidra(
    "sidra_10264_ocupados_grandes_grupos", table_code="10264", variable="4090",
    classifications={"2": "6794", "86": "95251", "526": "15349", "12064": "all"},
)
pop_ocup = ocup.set_index("Grandes grupos de ocupação no trabalho principal")["Valor"]
print("ocupados 14+ (Total):", f"{pop_ocup['Total']:,.0f}",
      "| soma dos grupos:", f"{pop_ocup.drop('Total').sum():,.0f}")
pop_ocup.drop("Total").map("{:,.0f}".format).to_frame("ocupados")

ocupados 14+ (Total): 88,679,494 | soma dos grupos: 88,679,494


,ocupados
Grandes grupos de ocupação no trabalho principal,
Diretores e gerentes,"4,090,824"
Profissionais das ciências e intelectuais,"10,530,417"
Técnicos e profissionais de nível médio,"6,960,883"
Trabalhadores de apoio administrativo,"7,128,482"
"Trabalhadores dos serviços, vendedores dos comércios e mercados","18,245,086"
"Trabalhadores qualificados da agropecuária, florestais, da caça e da pesca","4,459,242"
"Trabalhadores qualificados, operários e artesãos da construção, das artes mecânicas e outros ofícios","11,503,235"
Operadores de instalações e máquinas e montadores,"6,749,729"
Ocupações elementares,"15,219,319"


In [19]:
# Prévia das duas formas de tratar os mandatos políticos (decisão pendente)
#   A) políticos no grupo 1, como manda a COD
#   B) políticos fora da comparação (exibidos à parte); compara só quem declarou outra ocupação
pct_pop = 100 * pop_ocup.drop(["Total", "Ocupações mal definidas"]) / pop_ocup.drop(["Total", "Ocupações mal definidas"]).sum()

classif = eleitos26[eleitos26["COD_GRUPO"] != NC]
pct_A = 100 * classif["COD_GRUPO"].value_counts() / len(classif)
pct_B = 100 * classif.loc[~classif["POLITICO"], "COD_GRUPO"].value_counts() / (~classif["POLITICO"]).sum()

previa_ocup = pd.DataFrame({"% população ocupada": pct_pop, "% Câmara (A)": pct_A, "% Câmara (B)": pct_B}).fillna(0).round(1)
print(f"eleitos classificáveis: {len(classif)} de 513 | políticos: {int(classif['POLITICO'].sum())} "
      f"| base da opção B: {int((~classif['POLITICO']).sum())}")
print("obs.: 'Ocupações mal definidas' fica fora do denominador da população.")
previa_ocup.sort_values("% população ocupada", ascending=False)

eleitos classificáveis: 482 de 513 | políticos: 230 | base da opção B: 252
obs.: 'Ocupações mal definidas' fica fora do denominador da população.


,% população ocupada,% Câmara (A),% Câmara (B)
"Trabalhadores dos serviços, vendedores dos comércios e mercados",21.4,1.5,2.8
Ocupações elementares,17.8,0.2,0.4
"Trabalhadores qualificados, operários e artesãos da construção, das artes mecânicas e outros ofícios",13.5,0.0,0.0
Profissionais das ciências e intelectuais,12.3,32.6,62.3
Trabalhadores de apoio administrativo,8.3,0.2,0.4
Técnicos e profissionais de nível médio,8.1,0.6,1.2
Operadores de instalações e máquinas e montadores,7.9,0.0,0.0
"Trabalhadores qualificados da agropecuária, florestais, da caça e da pesca",5.2,2.7,5.2
Diretores e gerentes,4.8,60.4,24.2
"Membros das forças armadas, policiais, bombeiros militares",0.6,1.9,3.6


### 5.1 Decisão: mandatos políticos à parte (opção B)
Os eleitos que declararam ocupação de mandato político (deputado, vereador, senador, ministro de Estado)
ficam **fora da comparação com a COD** e aparecem num destaque separado. A comparação por grande grupo usa só
os eleitos com ocupação classificável e não política. Motivo: "deputado" como ocupação esconde a trajetória
profissional e, pela COD, jogaria todos no grupo 1, dominando o resultado.

In [20]:
ocup_classif = eleitos26[(eleitos26["COD_GRUPO"] != NC) & ~eleitos26["POLITICO"]]
ocup_resumo = {
    "eleitos": len(eleitos26),
    "mandato político (destaque)": int(eleitos26["POLITICO"].sum()),
    "não classificáveis": int((eleitos26["COD_GRUPO"] == NC).sum()),
    "base da comparação COD": len(ocup_classif),
}
ocup_resumo

{'eleitos': 513,
 'mandato político (destaque)': 230,
 'não classificáveis': 31,
 'base da comparação COD': 252}

## 6. Harmonização TSE × IBGE

Célula única de mapeamento: cada categoria do TSE vira a categoria correspondente do Censo 2022.
Categorias sem informação ("NÃO DIVULGÁVEL", "NÃO INFORMADO") viram `NaN` e saem dos denominadores — o
mesmo tratamento dado às 11.119 pessoas sem cor/raça declarada no Censo.

- **Gênero (TSE) → Sexo (Censo).** O TSE registra gênero autodeclarado; o Censo, sexo. Tratamos FEMININO = Mulheres
  e MASCULINO = Homens.
- **Cor/raça.** Categorias idênticas nas duas fontes. Agregado **"Pretos e pardos"** = Preta + Parda
  (convenção do IBGE para população negra).
- **Escolaridade.** As 7 categorias do TSE cabem sem ambiguidade nos 4 níveis do Censo.
- **Idade.** Faixas aplicadas igualmente aos eleitos (idade na posse) e à população 18+ (idade no Censo).

In [21]:
# ================= MAPEAMENTO TSE -> IBGE =================

# Gênero (TSE, autodeclarado) -> Sexo (Censo)
MAPA_GENERO = {
    "FEMININO": "Mulheres",
    "MASCULINO": "Homens",
    # "NÃO DIVULGÁVEL" -> NaN (sem informação)
}

# Cor ou raça: mesmas 5 categorias nas duas fontes
MAPA_RACA = {
    "BRANCA": "Branca",
    "PRETA": "Preta",
    "PARDA": "Parda",
    "AMARELA": "Amarela",
    "INDÍGENA": "Indígena",
    # "NÃO INFORMADO" / "NÃO DIVULGÁVEL" -> NaN
}
PRETOS_E_PARDOS = {"Preta", "Parda"}

# Grau de instrução (TSE, 7 categorias) -> Nível de instrução (Censo, 4 níveis)
ESC1 = "Sem instrução e fundamental incompleto"
ESC2 = "Fundamental completo e médio incompleto"
ESC3 = "Médio completo e superior incompleto"
ESC4 = "Superior completo"
MAPA_INSTRUCAO = {
    "LÊ E ESCREVE": ESC1,                    # sabe ler e escrever, sem escolaridade formal concluída
    "ENSINO FUNDAMENTAL INCOMPLETO": ESC1,
    "ENSINO FUNDAMENTAL COMPLETO": ESC2,
    "ENSINO MÉDIO INCOMPLETO": ESC2,
    "ENSINO MÉDIO COMPLETO": ESC3,
    "SUPERIOR INCOMPLETO": ESC3,
    "SUPERIOR COMPLETO": ESC4,
    # "NÃO DIVULGÁVEL" -> NaN
}
ORDEM_INSTRUCAO = [ESC1, ESC2, ESC3, ESC4]

# Faixas de idade (limite inferior inclusivo); a última é aberta
FAIXAS_IDADE = [18, 30, 40, 50, 60, 70]
ROTULOS_IDADE = ["18 a 29", "30 a 39", "40 a 49", "50 a 59", "60 a 69", "70 ou mais"]

def faixa_idade(idade: pd.Series) -> pd.Series:
    return pd.cut(idade, bins=FAIXAS_IDADE + [200], right=False, labels=ROTULOS_IDADE)

# ---------------- aplica no TSE ----------------
dep["SEXO_H"] = dep["DS_GENERO"].map(MAPA_GENERO)
dep["RACA_H"] = dep["DS_COR_RACA"].map(MAPA_RACA)
dep["PRETO_PARDO"] = dep["RACA_H"].isin(PRETOS_E_PARDOS).where(dep["RACA_H"].notna())
dep["INSTRUCAO_H"] = pd.Categorical(dep["DS_GRAU_INSTRUCAO"].map(MAPA_INSTRUCAO), categories=ORDEM_INSTRUCAO, ordered=True)
dep["FAIXA_IDADE"] = faixa_idade(dep["IDADE_POSSE"])

# Conferência: todo valor do TSE ficou mapeado, ou é uma categoria "sem informação" conhecida?
SEM_INFO = {"NÃO DIVULGÁVEL", "NÃO INFORMADO"}
for orig, harm in [("DS_GENERO", "SEXO_H"), ("DS_COR_RACA", "RACA_H"), ("DS_GRAU_INSTRUCAO", "INSTRUCAO_H")]:
    nao_mapeados = set(dep.loc[dep[harm].isna(), orig]) - SEM_INFO
    assert not nao_mapeados, f"{orig}: valores sem mapeamento {nao_mapeados}"

eleitos = dep[dep["ELEITO"]].copy()
eleitos26 = eleitos26.join(dep[["SEXO_H", "RACA_H", "PRETO_PARDO", "INSTRUCAO_H", "FAIXA_IDADE"]])

# Quantos ficaram sem informação, por ano (eleitos e candidatos do funil)
pd.DataFrame({
    f"{grupo} {ano}": {c: int(sub[c].isna().sum()) for c in ["SEXO_H", "RACA_H", "INSTRUCAO_H", "FAIXA_IDADE"]}
    for grupo, base_ in [("eleitos", dep[dep["ELEITO"]]), ("funil", dep[dep["CANDIDATO_FUNIL"]])]
    for ano, sub in base_.groupby("ANO_ELEICAO")
})

,eleitos 2018,eleitos 2022,eleitos 2026,funil 2018,funil 2022,funil 2026
SEXO_H,0,0,0,0,2,0
RACA_H,0,1,0,0,60,0
INSTRUCAO_H,0,0,0,0,2,0
FAIXA_IDADE,0,0,0,0,2,0


In [22]:
# ---------------- população (Censo 2022) nas mesmas categorias ----------------
# Sexo — população total
pop_sexo = sel(pop, **{"Cor_ou_raça": "Total", "Idade": "Total"}).query("Sexo != 'Total'").set_index("Sexo")["Valor"]

# Cor ou raça — população total, só quem declarou (denominador = soma das 5 categorias)
pop_raca = sel(pop, Sexo="Total", Idade="Total").query("`Cor ou raça` != 'Total'").set_index("Cor ou raça")["Valor"]

# Escolaridade — 25 anos ou mais (tabela 10061)
pop_esc = (sel(esc_tot, **{"Grupo_de_idade": "25 anos ou mais"}).query("`Nível de instrução` != 'Total'")
           .set_index("Nível de instrução")["Valor"].reindex(ORDEM_INSTRUCAO))

# Idade × sexo — 18 anos ou mais, idades simples agrupadas nas mesmas faixas (tabela 9606)
idades = sel(pop, **{"Cor_ou_raça": "Total"}).query("IDADE_SIMPLES and Sexo != 'Total' and IDADE_NUM >= 18").copy()
idades["FAIXA_IDADE"] = faixa_idade(idades["IDADE_NUM"])
pop_idade_sexo = idades.pivot_table(index="FAIXA_IDADE", columns="Sexo", values="Valor", aggfunc="sum", observed=False)

def pct(s: pd.Series) -> pd.Series:
    return (100 * s / s.sum()).round(1)

print("Sexo (%):", pct(pop_sexo).to_dict())
print("Cor ou raça (%, só declarados):", pct(pop_raca).to_dict(), "| pretos e pardos:",
      round(100 * pop_raca[list(PRETOS_E_PARDOS)].sum() / pop_raca.sum(), 1))
print("Escolaridade 25+ (%):", pct(pop_esc).to_dict())
print("\nIdade 18+ por sexo (pessoas):")
pop_idade_sexo.map("{:,.0f}".format)

Sexo (%): {'Homens': 48.5, 'Mulheres': 51.5}
Cor ou raça (%, só declarados): {'Branca': 43.5, 'Preta': 10.2, 'Amarela': 0.4, 'Parda': 45.3, 'Indígena': 0.6} | pretos e pardos: 55.5
Escolaridade 25+ (%): {'Sem instrução e fundamental incompleto': 35.2, 'Fundamental completo e médio incompleto': 14.0, 'Médio completo e superior incompleto': 32.3, 'Superior completo': 18.4}

Idade 18+ por sexo (pessoas):


Sexo,Homens,Mulheres
FAIXA_IDADE,,
18 a 29,"18,326,131","18,380,700"
30 a 39,"15,364,618","16,281,290"
40 a 49,"14,330,168","15,382,114"
50 a 59,"11,433,896","12,733,791"
60 a 69,"8,193,886","9,626,735"
70 ou mais,"6,031,867","8,261,002"


In [23]:
# Mesmas faixas entre os eleitos de 2026 (contagem) — só para conferir o encaixe; métricas vêm na etapa 7
pd.crosstab(eleitos26["FAIXA_IDADE"], eleitos26["SEXO_H"], margins=True, margins_name="Total")

SEXO_H,Homens,Mulheres,Total
FAIXA_IDADE,,,
18 a 29,17,4,21
30 a 39,58,27,85
40 a 49,123,37,160
50 a 59,100,26,126
60 a 69,77,13,90
70 ou mais,29,2,31
Total,404,109,513


## 7. Métricas

Decisão registrada: pirâmide etária em **faixas de 10 anos** (a primeira, 18 a 29, com 12 anos).

Para cada dimensão:
- `n_camara` e `%_camara` — eleitos de 2026 (denominador = eleitos com a informação);
- `%_pop` — população de referência (total para sexo e raça; 18+ para idade; 25+ para escolaridade;
  ocupados 14+ para ocupação);
- `dif_pp` — `%_camara − %_pop`, em pontos percentuais;
- `cadeiras_prop` — **Câmara proporcional**: quantas das 513 cadeiras o grupo teria se espelhasse a população.
  Arredondamento pelo **método dos maiores restos** (Hamilton), para que as cadeiras somem exatamente 513.

In [24]:
CADEIRAS = 513

def maiores_restos(pesos: pd.Series, total: int = CADEIRAS) -> pd.Series:
    """Distribui `total` cadeiras proporcionalmente aos pesos (método de Hamilton)."""
    cota = pesos / pesos.sum() * total
    inteiro = cota.astype(int)
    sobra = total - inteiro.sum()
    restos = (cota - inteiro).sort_values(ascending=False)
    inteiro.loc[restos.index[:sobra]] += 1
    return inteiro

def comparar(camara: pd.Series, populacao: pd.Series, ordem=None) -> pd.DataFrame:
    """camara: uma linha por eleito (categoria harmonizada); populacao: pessoas por categoria."""
    ordem = ordem or list(populacao.index)
    n = camara.value_counts().reindex(ordem, fill_value=0)
    pop_ = populacao.reindex(ordem)
    t = pd.DataFrame({
        "n_camara": n,
        "%_camara": 100 * n / n.sum(),
        "pop": pop_,
        "%_pop": 100 * pop_ / pop_.sum(),
    })
    t["dif_pp"] = t["%_camara"] - t["%_pop"]
    t["cadeiras_prop"] = maiores_restos(pop_)
    return t

def mostrar(t: pd.DataFrame) -> pd.DataFrame:
    out = t.copy()
    out["pop"] = out["pop"].map("{:,.0f}".format)
    return out.round(1)

In [25]:
# --- Gênero ---
m_genero = comparar(eleitos26["SEXO_H"], pop_sexo, ordem=["Mulheres", "Homens"])
mostrar(m_genero)

,n_camara,%_camara,pop,%_pop,dif_pp,cadeiras_prop
Mulheres,109,21.2,"104,548,325",51.5,-30.2,264
Homens,404,78.8,"98,532,431",48.5,30.2,249


In [26]:
# --- Cor ou raça (5 categorias) + agregado pretos e pardos ---
ORDEM_RACA = ["Branca", "Parda", "Preta", "Indígena", "Amarela"]
m_raca = comparar(eleitos26["RACA_H"], pop_raca, ordem=ORDEM_RACA)

pp = m_raca.loc[list(PRETOS_E_PARDOS)].sum()
pp["dif_pp"] = pp["%_camara"] - pp["%_pop"]
m_raca_agregado = pd.concat([m_raca, pp.to_frame("Pretos e pardos").T])
mostrar(m_raca_agregado)

,n_camara,%_camara,pop,%_pop,dif_pp,cadeiras_prop
Branca,374.0,72.9,"88,252,121",43.5,29.4,223.0
Parda,106.0,20.7,"92,083,286",45.3,-24.7,233.0
Preta,29.0,5.7,"20,656,458",10.2,-4.5,52.0
Indígena,3.0,0.6,"1,227,642",0.6,-0.0,3.0
Amarela,1.0,0.2,"850,130",0.4,-0.2,2.0
Pretos e pardos,135.0,26.3,"112,739,744",55.5,-29.2,285.0


In [27]:
# --- Sexo × cor/raça (base dos hemiciclos) ---
pop_sexo_raca = (sel(pop, Idade="Total").query("Sexo != 'Total' and `Cor ou raça` != 'Total'")
                 .set_index(["Sexo", "Cor ou raça"])["Valor"])
pop_sexo_raca.index = [f"{s} {r}" for s, r in pop_sexo_raca.index]
ORDEM_SEXO_RACA = [f"{s} {r}" for s in ["Mulheres", "Homens"] for r in ORDEM_RACA]
m_sexo_raca = comparar(eleitos26["SEXO_H"] + " " + eleitos26["RACA_H"], pop_sexo_raca, ordem=ORDEM_SEXO_RACA)
mostrar(m_sexo_raca)

,n_camara,%_camara,pop,%_pop,dif_pp,cadeiras_prop
Mulheres Branca,73,14.2,"46,463,555",22.9,-8.7,117
Mulheres Parda,21,4.1,"46,883,709",23.1,-19.0,118
Mulheres Preta,13,2.5,"10,130,438",5.0,-2.5,26
Mulheres Indígena,2,0.4,"616,574",0.3,0.1,2
Mulheres Amarela,0,0.0,"449,273",0.2,-0.2,1
Homens Branca,301,58.7,"41,788,566",20.6,38.1,106
Homens Parda,85,16.6,"45,199,577",22.3,-5.7,114
Homens Preta,16,3.1,"10,526,020",5.2,-2.1,27
Homens Indígena,1,0.2,"611,068",0.3,-0.1,1
Homens Amarela,1,0.2,"400,857",0.2,-0.0,1


In [28]:
# --- Idade: faixas (18+) e pirâmide por sexo ---
m_idade = comparar(eleitos26["FAIXA_IDADE"].astype(str), pop_idade_sexo.sum(axis=1).rename(index=str),
                   ordem=ROTULOS_IDADE)
display(mostrar(m_idade))

# Pirâmide: % de cada faixa×sexo sobre o total (eleitos e população 18+)
cam_piramide = pd.crosstab(eleitos26["FAIXA_IDADE"].astype(str), eleitos26["SEXO_H"]).reindex(ROTULOS_IDADE)
piramide = pd.concat({
    "%_camara": 100 * cam_piramide / cam_piramide.values.sum(),
    "%_pop": 100 * pop_idade_sexo.rename(index=str) / pop_idade_sexo.values.sum(),
}, axis=1).round(1)
print("idade na posse — mediana dos eleitos 2026:", eleitos26["IDADE_POSSE"].median())
piramide

,n_camara,%_camara,pop,%_pop,dif_pp,cadeiras_prop
FAIXA_IDADE,,,,,,
18 a 29,21,4.1,"36,706,831",23.8,-19.7,122
30 a 39,85,16.6,"31,645,908",20.5,-3.9,105
40 a 49,160,31.2,"29,712,282",19.3,11.9,99
50 a 59,126,24.6,"24,167,687",15.7,8.9,80
60 a 69,90,17.5,"17,820,621",11.5,6.0,59
70 ou mais,31,6.0,"14,292,869",9.3,-3.2,48


idade na posse — mediana dos eleitos 2026: 49.0


%_camara           %_pop         
SEXO_H        Homens Mulheres Homens Mulheres
FAIXA_IDADE                                  
18 a 29          3.3      0.8   11.9     11.9
30 a 39         11.3      5.3   10.0     10.5
40 a 49         24.0      7.2    9.3     10.0
50 a 59         19.5      5.1    7.4      8.3
60 a 69         15.0      2.5    5.3      6.2
70 ou mais       5.7      0.4    3.9      5.4

In [29]:
# --- Escolaridade: eleitos × população de 25+ ---
m_esc = comparar(eleitos26["INSTRUCAO_H"].astype(str), pop_esc, ordem=ORDEM_INSTRUCAO)
mostrar(m_esc)

,n_camara,%_camara,pop,%_pop,dif_pp,cadeiras_prop
Sem instrução e fundamental incompleto,5,1.0,"46,912,823",35.2,-34.3,181
Fundamental completo e médio incompleto,11,2.1,"18,666,167",14.0,-11.9,72
Médio completo e superior incompleto,76,14.8,"43,044,784",32.3,-17.5,166
Superior completo,421,82.1,"24,484,247",18.4,63.7,94


In [30]:
# --- Ocupação (opção B): eleitos não políticos e classificáveis × ocupados 14+ ---
pop_ocup_cmp = pop_ocup.drop(["Total", "Ocupações mal definidas"])
m_ocup = comparar(ocup_classif["COD_GRUPO"], pop_ocup_cmp)
display(mostrar(m_ocup.sort_values("%_pop", ascending=False)))

# Top ocupações declaradas (todas as 513, inclusive políticas) — parte descritiva
top_ocup = eleitos26["DS_OCUPACAO"].value_counts().head(10).to_frame("eleitos")
top_ocup["%"] = (100 * top_ocup["eleitos"] / len(eleitos26)).round(1)
top_ocup

,n_camara,%_camara,pop,%_pop,dif_pp,cadeiras_prop
"Trabalhadores dos serviços, vendedores dos comércios e mercados",7,2.8,"18,245,086",21.4,-18.6,110
Ocupações elementares,1,0.4,"15,219,319",17.8,-17.4,91
"Trabalhadores qualificados, operários e artesãos da construção, das artes mecânicas e outros ofícios",0,0.0,"11,503,235",13.5,-13.5,69
Profissionais das ciências e intelectuais,157,62.3,"10,530,417",12.3,50.0,63
Trabalhadores de apoio administrativo,1,0.4,"7,128,482",8.3,-7.9,43
Técnicos e profissionais de nível médio,3,1.2,"6,960,883",8.1,-7.0,42
Operadores de instalações e máquinas e montadores,0,0.0,"6,749,729",7.9,-7.9,40
"Trabalhadores qualificados da agropecuária, florestais, da caça e da pesca",13,5.2,"4,459,242",5.2,-0.1,27
Diretores e gerentes,61,24.2,"4,090,824",4.8,19.4,25
"Membros das forças armadas, policiais, bombeiros militares",9,3.6,"525,275",0.6,3.0,3


,eleitos,%
DS_OCUPACAO,,
DEPUTADA/DEPUTADO,201,39.2
EMPRESÁRIA/EMPRESÁRIO,61,11.9
ADVOGADA/ADVOGADO,55,10.7
MÉDICA/MÉDICO,30,5.8
VEREADORA/VEREADOR,27,5.3
ADMINISTRADORA/ADMINISTRADOR,10,1.9
SERVIDOR PÚBLICO ESTADUAL,9,1.8
OUTROS,9,1.8
PROFESSORA/PROFESSOR DE ENSINO SUPERIOR,8,1.6


In [31]:
# --- Evolução 2018 → 2022 → 2026 entre os eleitos ---
evolucao = eleitos.groupby("ANO_ELEICAO").agg(
    mulheres=("SEXO_H", lambda s: (s == "Mulheres").sum()),
    com_sexo=("SEXO_H", "count"),
    pretos_pardos=("PRETO_PARDO", lambda s: (s == True).sum()),
    com_raca=("RACA_H", "count"),
)
evolucao["%_mulheres"] = (100 * evolucao["mulheres"] / evolucao["com_sexo"]).round(1)
evolucao["%_pretos_pardos"] = (100 * evolucao["pretos_pardos"] / evolucao["com_raca"]).round(1)
evolucao

,mulheres,com_sexo,pretos_pardos,com_raca,%_mulheres,%_pretos_pardos
ANO_ELEICAO,,,,,,
2018,77,513,123,513,15.0,24.0
2022,90,513,139,512,17.5,27.1
2026,109,513,135,513,21.2,26.3


In [32]:
# --- Funil candidatos (sem #NULO) → eleitos, por ano ---
dep["GRUPO_RACA"] = dep["RACA_H"].map(
    lambda r: "Pretos e pardos" if r in PRETOS_E_PARDOS else ("Brancos" if r == "Branca" else ("Outros" if pd.notna(r) else None))
)
dep["SEXO_RACA"] = dep["SEXO_H"] + " " + dep["GRUPO_RACA"].str.lower()

def funil(var: str) -> pd.DataFrame:
    base_ = dep[dep["CANDIDATO_FUNIL"] & dep[var].notna()]
    t = base_.groupby(["ANO_ELEICAO", var]).agg(candidatos=("SQ_CANDIDATO", "size"), eleitos=("ELEITO", "sum"))
    tot = t.groupby(level=0).transform("sum")
    t["%_candidatos"] = 100 * t["candidatos"] / tot["candidatos"]
    t["%_eleitos"] = 100 * t["eleitos"] / tot["eleitos"]
    t["taxa_sucesso_%"] = 100 * t["eleitos"] / t["candidatos"]
    return t.round(1)

f_sexo, f_raca, f_sexo_raca = funil("SEXO_H"), funil("GRUPO_RACA"), funil("SEXO_RACA")
display(f_sexo)
display(f_raca)
f_sexo_raca.loc[ANO_PRINCIPAL]

candidatos  eleitos  %_candidatos  %_eleitos  taxa_sucesso_%
ANO_ELEICAO SEXO_H                                                                
2018        Homens          5528      436          68.4       85.0             7.9
            Mulheres        2555       77          31.6       15.0             3.0
2022        Homens          6470      423          64.9       82.5             6.5
            Mulheres        3498       90          35.1       17.5             2.6
2026        Homens          4655      404          63.0       78.8             8.7
            Mulheres        2738      109          37.0       21.2             4.0

candidatos  eleitos  %_candidatos  %_eleitos  taxa_sucesso_%
ANO_ELEICAO GRUPO_RACA                                                                   
2018        Brancos                4651      387          57.5       75.4             8.3
            Outros                   93        3           1.2        0.6             3.2
            Pretos e pardos        3339      123          41.3       24.0             3.7
2022        Brancos                5034      366          50.8       71.5             7.3
            Outros                   95        7           1.0        1.4             7.4
            Pretos e pardos        4781      139          48.2       27.1             2.9
2026        Brancos                3853      374          52.1       72.9             9.7
            Outros                  111        4           1.5        0.8             3.6
            Pretos e pardos        3429      135          46.4       26.3             3.9

,candidatos,eleitos,%_candidatos,%_eleitos,taxa_sucesso_%
SEXO_RACA,,,,,
Homens brancos,2518,301,34.1,58.7,12.0
Homens outros,64,2,0.9,0.4,3.1
Homens pretos e pardos,2073,101,28.0,19.7,4.9
Mulheres brancos,1335,73,18.1,14.2,5.5
Mulheres outros,47,2,0.6,0.4,4.3
Mulheres pretos e pardos,1356,34,18.3,6.6,2.5


## 8. Exportação — um JSON por gráfico em `output/`

Cada arquivo é gravado em dois lugares: `output/` (deste projeto) e a pasta `dados/` da publicação no
gerador de conteúdo (`../desviopadrao-slides/src/posts/2026-11-a-camara-parece-com-o-brasil/dados/`).
O `metodologia.json` traz também o bloco `meta` (título, período, fonte, @) que o gerador usa no rodapé dos slides.

| Arquivo | Slide | Conteúdo |
|---|---|---|
| `hemiciclo_proporcional.json` | 1 — capa | 513 cadeiras por sexo × cor/raça, se a Câmara espelhasse a população |
| `hemiciclo_real.json` | 2 — Câmara real | 513 eleitos de 2026 por sexo × cor/raça |
| `genero.json` | 3 | Câmara × população, dif. em pp, cadeiras proporcionais |
| `raca.json` | 4 | idem, 5 categorias + agregado pretos e pardos (barras divergentes) |
| `idade_piramide.json` | 5 | % por faixa × sexo, eleitos × população 18+ |
| `escolaridade.json` | 6 | Câmara × população 25+ |
| `ocupacao.json` | 6 | destaque dos mandatos políticos + comparação COD + top ocupações declaradas |
| `evolucao.json` | 7 | % mulheres e % pretos e pardos, 2018 → 2026 |
| `funil.json` | (apoio) | candidatos → eleitos por sexo, raça e sexo × raça, 3 anos |
| `metodologia.json` | 7 | fontes, datas e notas metodológicas |

Convenções: percentuais com 1 casa decimal; `dif_pp` = % Câmara − % população; rótulos em português,
com `id` sem acento para uso no código do front.

In [33]:
import json
import unicodedata
from datetime import date

def slug(texto: str) -> str:
    s = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")

def r1(x) -> float:
    return round(float(x), 1) + 0.0  # + 0.0 evita '-0.0' no JSON

# Pasta da publicação no gerador de conteúdo (projeto irmão desviopadrao-slides)
SLUG_POST = "2026-11-a-camara-parece-com-o-brasil"
DIR_POST = RAIZ.parent / "desviopadrao-slides" / "src" / "posts" / SLUG_POST / "dados"
DIR_POST.mkdir(parents=True, exist_ok=True)

def salvar(nome: str, obj) -> None:
    """Grava o JSON em output/ e na pasta dados/ da publicação."""
    texto = json.dumps(obj, ensure_ascii=False, indent=2)
    for pasta in (DIR_OUT, DIR_POST):
        (pasta / nome).write_text(texto, encoding="utf-8")
    print(f"ok  {nome}  ({len(texto.encode()):,} bytes) -> output/ e gerador")

def linhas_comparacao(t: pd.DataFrame, rotulos: dict | None = None) -> list[dict]:
    rotulos = rotulos or {}
    return [{
        "id": slug(str(k)),
        "grupo": rotulos.get(k, str(k)),
        "camara_n": int(v["n_camara"]),
        "camara_pct": r1(v["%_camara"]),
        "populacao_pct": r1(v["%_pop"]),
        "dif_pp": r1(v["dif_pp"]),
        "cadeiras_proporcionais": int(v["cadeiras_prop"]),
    } for k, v in t.iterrows()]

# Rótulos com concordância para sexo × raça
ROTULO_SEXO_RACA = {
    "Mulheres Branca": "Mulheres brancas", "Mulheres Parda": "Mulheres pardas", "Mulheres Preta": "Mulheres pretas",
    "Mulheres Indígena": "Mulheres indígenas", "Mulheres Amarela": "Mulheres amarelas",
    "Homens Branca": "Homens brancos", "Homens Parda": "Homens pardos", "Homens Preta": "Homens pretos",
    "Homens Indígena": "Homens indígenas", "Homens Amarela": "Homens amarelos",
}
assert set(ROTULO_SEXO_RACA) == set(m_sexo_raca.index)

In [34]:
# --- Slides 1 e 2: hemiciclos (sexo × raça) ---
def hemiciclo(coluna: str, titulo: str) -> dict:
    grupos = [{
        "id": slug(k), "sexo": k.split(" ")[0], "raca": k.split(" ", 1)[1],
        "rotulo": ROTULO_SEXO_RACA[k], "cadeiras": int(v[coluna]),
    } for k, v in m_sexo_raca.iterrows()]
    assert sum(g["cadeiras"] for g in grupos) == CADEIRAS
    return {"titulo": titulo, "total_cadeiras": CADEIRAS, "ordem": "mulheres primeiro; raça: branca, parda, preta, indígena, amarela", "grupos": grupos}

salvar("hemiciclo_proporcional.json", hemiciclo("cadeiras_prop", "Se a Câmara fosse o Brasil — cadeiras proporcionais à população (Censo 2022)"))
salvar("hemiciclo_real.json", hemiciclo("n_camara", "Câmara dos Deputados eleita em 2026 (legislatura 2027–2031)"))

# --- Slide 3: gênero ---
salvar("genero.json", {"referencia_populacao": "população residente total, Censo 2022", "dados": linhas_comparacao(m_genero)})

# --- Slide 4: raça ---
salvar("raca.json", {
    "referencia_populacao": "população residente total que declarou cor ou raça, Censo 2022",
    "dados": linhas_comparacao(m_raca),
    "agregado_pretos_e_pardos": linhas_comparacao(m_raca_agregado.loc[["Pretos e pardos"]])[0],
})

# --- Slide 5: pirâmide etária ---
salvar("idade_piramide.json", {
    "referencia_populacao": "população residente de 18 anos ou mais, Censo 2022 (idade em 2022)",
    "referencia_camara": "idade dos eleitos na posse (01/02/2027)",
    "mediana_idade_camara": float(eleitos26["IDADE_POSSE"].median()),
    "faixas": [{
        "faixa": f,
        "camara_homens_pct": r1(piramide.loc[f, ("%_camara", "Homens")]),
        "camara_mulheres_pct": r1(piramide.loc[f, ("%_camara", "Mulheres")]),
        "populacao_homens_pct": r1(piramide.loc[f, ("%_pop", "Homens")]),
        "populacao_mulheres_pct": r1(piramide.loc[f, ("%_pop", "Mulheres")]),
    } for f in ROTULOS_IDADE],
    "totais_por_faixa": linhas_comparacao(m_idade),
})

# --- Slide 6: escolaridade ---
salvar("escolaridade.json", {"referencia_populacao": "pessoas de 25 anos ou mais, Censo 2022", "dados": linhas_comparacao(m_esc)})

ok  hemiciclo_proporcional.json  (1,714 bytes) -> output/ e gerador
ok  hemiciclo_real.json  (1,691 bytes) -> output/ e gerador
ok  genero.json  (485 bytes) -> output/ e gerador
ok  raca.json  (1,307 bytes) -> output/ e gerador
ok  idade_piramide.json  (2,494 bytes) -> output/ e gerador
ok  escolaridade.json  (1,082 bytes) -> output/ e gerador


In [35]:
# --- Slide 6: ocupação ---
ROTULO_COD_CURTO = {
    G0: "Militares e policiais militares",
    G1: "Diretores e gerentes",
    G2: "Profissionais de nível superior",
    G3: "Técnicos de nível médio",
    G4: "Apoio administrativo",
    G5: "Serviços e comércio",
    G6: "Agropecuária",
    "Trabalhadores qualificados, operários e artesãos da construção, das artes mecânicas e outros ofícios": "Operários e artesãos",
    "Operadores de instalações e máquinas e montadores": "Operadores de máquinas",
    G9: "Ocupações elementares",
}
dados_ocup = linhas_comparacao(m_ocup.sort_values("%_pop", ascending=False), ROTULO_COD_CURTO)
for d, nome_ibge in zip(dados_ocup, m_ocup.sort_values("%_pop", ascending=False).index):
    d["nome_cod"] = nome_ibge

salvar("ocupacao.json", {
    "destaque_mandato_politico": {
        "eleitos": int(ocup_resumo["mandato político (destaque)"]),
        "total": CADEIRAS,
        "pct": r1(100 * ocup_resumo["mandato político (destaque)"] / CADEIRAS),
        "ocupacoes": sorted(OCUPACOES_POLITICAS),
    },
    "base_comparacao": {k: int(v) for k, v in ocup_resumo.items()},
    "referencia_populacao": "pessoas de 14 anos ou mais ocupadas na semana de referência, Censo 2022 (exclui 'Ocupações mal definidas')",
    "nota": "cadeiras_proporcionais distribuem as 513 cadeiras pela estrutura ocupacional da população; percentuais da Câmara são sobre os eleitos não políticos e classificáveis",
    "dados": dados_ocup,
    "top_ocupacoes_declaradas": [
        {"ocupacao": o, "eleitos": int(v["eleitos"]), "pct": r1(v["%"])} for o, v in top_ocup.iterrows()
    ],
})

# --- Slide 7: evolução ---
salvar("evolucao.json", {
    "nota": "percentual sobre eleitos com a informação declarada",
    "dados": [{
        "ano": int(a), "legislatura": f"{a + 1}–{a + 5}",
        "mulheres_n": int(v["mulheres"]), "mulheres_pct": r1(v["%_mulheres"]),
        "pretos_pardos_n": int(v["pretos_pardos"]), "pretos_pardos_pct": r1(v["%_pretos_pardos"]),
    } for a, v in evolucao.iterrows()],
    "populacao_referencia": {
        "mulheres_pct": r1(m_genero.loc["Mulheres", "%_pop"]),
        "pretos_pardos_pct": r1(m_raca_agregado.loc["Pretos e pardos", "%_pop"]),
        "fonte": "Censo 2022",
    },
})

# --- Funil ---
ROTULO_FUNIL = {
    "Homens brancos": "Homens brancos", "Homens pretos e pardos": "Homens pretos e pardos", "Homens outros": "Homens de outras raças",
    "Mulheres brancos": "Mulheres brancas", "Mulheres pretos e pardos": "Mulheres pretas e pardas", "Mulheres outros": "Mulheres de outras raças",
    "Brancos": "Brancos", "Pretos e pardos": "Pretos e pardos", "Outros": "Outras raças (amarela e indígena)",
    "Homens": "Homens", "Mulheres": "Mulheres",
}
def linhas_funil(t: pd.DataFrame) -> list[dict]:
    return [{
        "ano": int(ano), "id": slug(ROTULO_FUNIL[g]), "grupo": ROTULO_FUNIL[g],
        "candidatos": int(v["candidatos"]), "eleitos": int(v["eleitos"]),
        "candidatos_pct": r1(v["%_candidatos"]), "eleitos_pct": r1(v["%_eleitos"]),
        "taxa_sucesso_pct": r1(v["taxa_sucesso_%"]),
    } for (ano, g), v in t.iterrows()]

salvar("funil.json", {
    "definicao_candidatos": "candidaturas a deputado federal com votos totalizados (DS_SIT_TOT_TURNO diferente de #NULO)",
    "por_sexo": linhas_funil(f_sexo),
    "por_raca": linhas_funil(f_raca),
    "por_sexo_raca": linhas_funil(f_sexo_raca),
})

ok  ocupacao.json  (4,981 bytes) -> output/ e gerador
ok  evolucao.json  (749 bytes) -> output/ e gerador
ok  funil.json  (7,944 bytes) -> output/ e gerador


In [36]:
# --- Metodologia e fontes ---
NOTAS_METODOLOGICAS = [
    "Câmara de referência: os 513 deputados federais eleitos em 2026 (legislatura 2027–2031), segundo o TSE. Licenças, cassações e decisões judiciais posteriores podem alterar quem exerce o mandato.",
    "Gênero, cor/raça e ocupação são autodeclarados no registro de candidatura (TSE); sexo e cor/raça são autodeclarados no Censo 2022. Gênero FEMININO/MASCULINO do TSE foi comparado a Mulheres/Homens do Censo.",
    "Gênero e cor/raça: comparação com a população residente total. Cor/raça: denominador = quem declarou (exclui 11.119 pessoas sem declaração no Censo).",
    "Idade: idade dos eleitos na posse (01/02/2027) × população de 18 anos ou mais no Censo (2022), em faixas de 10 anos.",
    "Escolaridade: comparação com a população de 25 anos ou mais. As 7 categorias do TSE foram agrupadas nos 4 níveis do Censo.",
    "Ocupação: mapeamento próprio da lista do TSE para os grandes grupos da COD (sem tabela oficial de correspondência). Mandatos políticos (deputado, vereador, senador, ministro) ficam à parte; servidores sem cargo, aposentados, estudantes, donas de casa e 'outros' não são classificáveis.",
    "Câmara proporcional: 513 cadeiras distribuídas pelo método dos maiores restos (Hamilton).",
    "Funil: candidatos = candidaturas com votos totalizados (exclui #NULO). Em 2026, o TSE não publicou a situação de aptidão da candidatura.",
    "Na base do TSE, um eleito de 2026 aparece com 20 anos na data da posse (abaixo do mínimo constitucional de 21); mantido como publicado.",
    "Pretos e pardos: agregado de Preta + Parda, convenção do IBGE para população negra.",
]
salvar("metodologia.json", {
    # bloco no formato `meta` do gerador de conteúdo (rodapé dos slides)
    "meta": {
        "titulo": "Se a Câmara realmente representasse o Brasil?",
        "periodo": "legislatura 2027–2031",
        "extraido_em": date.today().strftime("%Y-%m"),
        "fonte": "TSE (candidaturas 2018–2026) e IBGE (Censo 2022)",
        "handle": "@desviopadrao.info",
    },
    "titulo": "Se a Câmara realmente representasse o Brasil?",
    "autor": "Desvio Padrão (@desviopadrao.info)",
    "data_extracao": date.today().isoformat(),
    "fontes": [
        {"nome": "TSE — Candidatos (consulta_cand)", "anos": ANOS, "url": URL_TSE.format(ano="{ano}"),
         "data_geracao": {int(a): df["DT_GERACAO"].iloc[0] for a, df in cand_raw.items()}},
        {"nome": "IBGE — Censo Demográfico 2022 (SIDRA)", "tabelas": {
            "9606": "População residente, por cor ou raça, segundo o sexo e a idade",
            "10061": "Pessoas de 18 anos ou mais de idade, por nível de instrução, segundo os grupos de idade, o sexo e a cor ou raça",
            "10264": "Pessoas de 14 anos ou mais ocupadas, por sexo, cor ou raça, contribuição previdenciária e grandes grupos de ocupação",
        }, "url": "https://sidra.ibge.gov.br"},
    ],
    "notas": NOTAS_METODOLOGICAS,
})

ok  metodologia.json  (2,908 bytes) -> output/ e gerador


In [37]:
# Validação final: todos os JSONs abrem e as cadeiras somam 513 onde devem
for arq in sorted(DIR_OUT.glob("*.json")):
    obj = json.loads(arq.read_text(encoding="utf-8"))
    somas = []
    for chave in ("dados", "grupos"):
        if chave in obj and obj[chave] and "cadeiras_proporcionais" in obj[chave][0]:
            somas.append(sum(d["cadeiras_proporcionais"] for d in obj[chave]))
        if chave in obj and obj[chave] and "cadeiras" in obj[chave][0]:
            somas.append(sum(d["cadeiras"] for d in obj[chave]))
    assert all(s == CADEIRAS for s in somas), (arq.name, somas)
    print(f"{arq.name:28s} chaves: {list(obj)}  somas de cadeiras: {somas}")

escolaridade.json            chaves: ['referencia_populacao', 'dados']  somas de cadeiras: [513]
evolucao.json                chaves: ['nota', 'dados', 'populacao_referencia']  somas de cadeiras: []
funil.json                   chaves: ['definicao_candidatos', 'por_sexo', 'por_raca', 'por_sexo_raca']  somas de cadeiras: []
genero.json                  chaves: ['referencia_populacao', 'dados']  somas de cadeiras: [513]
hemiciclo_proporcional.json  chaves: ['titulo', 'total_cadeiras', 'ordem', 'grupos']  somas de cadeiras: [513]
hemiciclo_real.json          chaves: ['titulo', 'total_cadeiras', 'ordem', 'grupos']  somas de cadeiras: [513]
idade_piramide.json          chaves: ['referencia_populacao', 'referencia_camara', 'mediana_idade_camara', 'faixas', 'totais_por_faixa']  somas de cadeiras: []
metodologia.json             chaves: ['meta', 'titulo', 'autor', 'data_extracao', 'fontes', 'notas']  somas de cadeiras: []
ocupacao.json                chaves: ['destaque_mandato_politico', 'base